# Phase 3+4 integration — n=10 verification with Saighi A_k self-inhibition (Colab, parallel)

Runs `experiments/19_phase34_integrated.py` for **10 seeds** {17, 11, 23, 1, 2, 3, 5, 7, 13, 19} on CUDA with `--inhibition-gain 0.01 --n-cues 3000`.

**Key design choice:** parent process never touches CUDA. Earlier debugging revealed workers hung because the parent kernel held the GPU (Colab's GPU was in `EXCLUSIVE_PROCESS` compute mode or otherwise blocked workers from grabbing CUDA while the parent held it). All `device='cuda'` sanity checks moved to CPU; the codebook is loaded on CPU in the parent just to verify the file. Each worker subprocess initializes its own CUDA context fresh.

Per-seed runtime is ~17 min sequential because workload is CPU-bound. Launching all 10 as subprocesses cuts wall-clock to ~25–40 min total.

**Why n=10 / n_cues=3000:** Report 032 showed n=5 has been *sample-lucky* twice. Report 033: patterns reach equilibrium at cue ~1200, so n_cues=1500 cuts off in the transient.

**Verifies report 034's seed-1 result.** Seed 1 showed top1 +0.009, R@10 +0.022, cap_t05 +0.022.

**Watching:** 8/10+ seeds with ΔR@10 ≥ 0; seed 23 (persistent negative outlier); Δtop1 (blocker #6'); Δcap_t05 (blocker #3); A_k accumulation in `death_diag`.

**If anything goes wrong:** Cell 5c is an emergency-kill cell, Cell 5d runs py-spy on the first stuck worker.

In [ ]:
# 1. Clone the repo.
%cd /content
!rm -rf Neuro-AI
!git clone https://github.com/Dypatterson/Neuro-AI.git
%cd Neuro-AI
!git log --oneline -5

In [ ]:
# 2. Mount Drive and copy the codebook into place.
from google.colab import drive
drive.mount('/content/drive')

import shutil, os
src = '/content/drive/MyDrive/neuro-ai/phase3c_codebook_reconstruction.pt'
dst_dir = 'reports/phase3c_reconstruction'
os.makedirs(dst_dir, exist_ok=True)
shutil.copy(src, f'{dst_dir}/phase3c_codebook_reconstruction.pt')
!ls -lh {dst_dir}/phase3c_codebook_reconstruction.pt

In [ ]:
# 3. Install deps. Colab ships torch+CUDA; we need datasets (wikitext) and py-spy (for debugging stuck workers).
!pip install -q datasets py-spy

In [ ]:
# 4. CPU-ONLY sanity check. The parent kernel deliberately does NOT touch CUDA —
# Colab's GPU compute mode blocks worker subprocesses from grabbing CUDA while
# the parent holds it. The workers each init their own CUDA context fresh.
import os
print('cpu count:', os.cpu_count())

import sys; sys.path.insert(0, 'src')
from energy_memory.phase2.persistence import load_codebook
cb = load_codebook('reports/phase3c_reconstruction/phase3c_codebook_reconstruction.pt', device='cpu')
print('codebook (parent-side CPU load):', cb.shape, cb.dtype, cb.device)

# Verify A_k mechanism is present in this checkout (CPU-only — workers test on CUDA later).
from energy_memory.phase4.consolidation import ConsolidationConfig, ConsolidationState
cfg = ConsolidationConfig(m=4, alpha=0.25, inhibition_gain=0.01)
s = ConsolidationState(cfg, device='cpu')
s.add_pattern()
s.accumulate_inhibition(0)
assert abs(s.A[0].item() - 0.01) < 1e-6, 'A_k not wired — code is stale, abort!'
print('A_k mechanism present (CPU sanity check):', s.A[0].item())

# Free the parent's tensors before launching workers.
del cb, s, cfg
import gc; gc.collect()

In [ ]:
# 4b. Pre-warm the HuggingFace wikitext cache so 10 subprocesses don't race on download.
print('warming wikitext cache...')
from energy_memory.phase2.corpus import load_corpus_splits
from pathlib import Path
splits = load_corpus_splits('wikitext', Path('.'), wikitext_name='wikitext-2-raw-v1')
print('  train:', len(splits['train']), 'rows')
print('  validation:', len(splits['validation']), 'rows')
print('cache warmed.')
del splits; gc.collect()

In [ ]:
# 4c. Quick GPU info (does NOT init CUDA from this notebook). Just nvidia-smi.
print('=== GPU info ===')
!nvidia-smi --query-gpu=name,memory.total,compute_mode --format=csv
print()
print('=== Current GPU processes (should be empty before we launch workers) ===')
!nvidia-smi --query-compute-apps=pid,process_name,used_memory --format=csv

In [ ]:
# 5. PARALLEL launch: 10 seeds as independent subprocesses.
# Each subprocess initializes its own CUDA context fresh (parent never touched CUDA).
#
# Knob: PARALLEL_BATCH_SIZE controls how many run at once.
#   10 = all simultaneous (fastest if CUDA + CPU can handle it)
#   5  = halves; 2× wall-clock but less contention
#   2  = pairs; ~5× wall-clock
#   1  = fully sequential (safe fallback, ~170 min)
PARALLEL_BATCH_SIZE = 10

import subprocess, os, time
from pathlib import Path

os.environ['PYTHONPATH'] = '/content/Neuro-AI/src'
RUN_TAG = 'saighi_n3k'
SEEDS = [17, 11, 23, 1, 2, 3, 5, 7, 13, 19]
log_root = Path(f'reports/phase34_{RUN_TAG}_colab')
log_root.mkdir(parents=True, exist_ok=True)

def launch(seed):
    out_dir = f'reports/phase34_{RUN_TAG}_seed{seed}'
    Path(out_dir).mkdir(parents=True, exist_ok=True)
    log_path = log_root / f'seed{seed}.log'
    logf = open(log_path, 'w')
    proc = subprocess.Popen(
        ['python', 'experiments/19_phase34_integrated.py',
         '--updater-kind', 'hebbian',
         '--seed', str(seed),
         '--n-cues', '3000',
         '--checkpoint-every', '500',
         '--success-threshold', '0.3',
         '--death-threshold', '0.05',
         '--death-window', '10',
         '--inhibition-gain', '0.01',
         '--inhibition-decay', '0.0',
         '--no-reencode-discovered',
         '--output-dir', out_dir],
        stdout=logf, stderr=subprocess.STDOUT,
    )
    return proc, logf, out_dir

def run_batch(seeds_batch, t0):
    procs = {seed: launch(seed) for seed in seeds_batch}
    print(f'  launched batch of {len(seeds_batch)}: {seeds_batch}  pids={[p[0].pid for p in procs.values()]}')
    remaining = dict(procs)
    while remaining:
        done_this_round = []
        for seed, (proc, logf, out_dir) in remaining.items():
            rc = proc.poll()
            if rc is not None:
                logf.close()
                elapsed = (time.time() - t0) / 60
                ok = 'OK' if rc == 0 else f'FAILED (exit={rc})'
                json_exists = Path(out_dir, 'phase34_results.json').exists()
                print(f'  [{elapsed:5.1f} min] seed {seed:>2}: {ok}  json={json_exists}')
                done_this_round.append(seed)
        for seed in done_this_round:
            del remaining[seed]
        if remaining:
            time.sleep(30)

# Split into batches of PARALLEL_BATCH_SIZE.
batches = [SEEDS[i:i+PARALLEL_BATCH_SIZE] for i in range(0, len(SEEDS), PARALLEL_BATCH_SIZE)]
print(f'Running {len(SEEDS)} seeds in {len(batches)} batches of up to {PARALLEL_BATCH_SIZE} ({time.strftime("%H:%M:%S")})')

t0 = time.time()
for i, batch in enumerate(batches, 1):
    print(f'\n=== Batch {i}/{len(batches)} ({time.strftime("%H:%M:%S")}) ===')
    run_batch(batch, t0)

print(f'\nALL DONE in {(time.time()-t0)/60:.1f} min')
print()
print('=== Final GPU state ===')
!nvidia-smi --query-gpu=memory.used,memory.total,utilization.gpu --format=csv

In [ ]:
# 5b. PROGRESS CHECK — run this in a separate cell ANY TIME during a long run
# to see where each worker is. Cell 5 itself only prints when a seed completes.
import subprocess
from pathlib import Path

log_root = Path('reports/phase34_saighi_n3k_colab')
print('=== alive workers ===')
!ps aux | grep "experiments/19" | grep -v grep | awk '{print $2, $9, $10}'
print()
print('=== GPU snapshot ===')
!nvidia-smi --query-gpu=memory.used,memory.total,utilization.gpu --format=csv
print()
!nvidia-smi --query-compute-apps=pid,used_memory --format=csv
print()
print('=== last log line per seed ===')
for seed in [17, 11, 23, 1, 2, 3, 5, 7, 13, 19]:
    log = log_root / f'seed{seed}.log'
    if not log.exists():
        print(f'seed {seed:>2}: (no log)')
        continue
    try:
        line = subprocess.check_output(['tail', '-1', str(log)]).decode().strip()
    except Exception:
        line = '(empty)'
    size = log.stat().st_size
    print(f'seed {seed:>2}: {size:>7}b  {line[:110]}')

In [ ]:
# 5c. EMERGENCY: kill all worker subprocesses. Run this if cell 5 stalls and you want to retry.
import subprocess, signal, os
killed = 0
for line in subprocess.check_output(['ps', '-eo', 'pid,cmd']).decode().splitlines():
    if 'experiments/19' in line and 'grep' not in line:
        try:
            pid = int(line.split()[0])
            os.kill(pid, signal.SIGKILL)
            print(f'  killed {pid}')
            killed += 1
        except Exception as e:
            print(f'  {pid}: {e}')
print(f'killed {killed} workers')
# Also interrupt the kernel from the runtime menu (Runtime → Interrupt) if cell 5 is still running.

In [ ]:
# 5d. PY-SPY DUMP — if workers are alive but stuck, run this to see where.
# Pass the PID of any worker (e.g. from cell 5b's 'alive workers' list).
import subprocess
PID = None  # ← put a worker PID here, e.g. 11730
if PID is None:
    print('Set PID at the top of this cell to a worker PID from cell 5b, then re-run.')
else:
    !py-spy dump --pid {PID}

In [ ]:
# 6. Copy results back to Drive. Claude will pull from there for analysis.
import shutil, os
RUN_TAG = 'saighi_n3k'
SEEDS = [17, 11, 23, 1, 2, 3, 5, 7, 13, 19]
dst = '/content/drive/MyDrive/neuro-ai/results'
os.makedirs(dst, exist_ok=True)
for seed in SEEDS:
    src = f'reports/phase34_{RUN_TAG}_seed{seed}'
    if os.path.isdir(src):
        shutil.copytree(src, f'{dst}/phase34_{RUN_TAG}_seed{seed}', dirs_exist_ok=True)
shutil.copytree(f'reports/phase34_{RUN_TAG}_colab',
                f'{dst}/phase34_{RUN_TAG}_colab', dirs_exist_ok=True)
print('results in', dst)
!ls {dst} | grep -E 'saighi_n3k'

In [ ]:
# 7. In-notebook summary table (per-seed + n=10 aggregate). Claude will re-do with raw JSONs but this lets you spot anomalies live.
import json
from pathlib import Path
import statistics as st

RUN_TAG = 'saighi_n3k'
SEEDS = [17, 11, 23, 1, 2, 3, 5, 7, 13, 19]
rows = []
print(f'{"seed":>4} {"cond":>16} {"top1":>8} {"R@10":>8} {"cap_t05":>8} {"cands":>6} {"cons":>5} {"deaths":>6} {"A_max":>8} {"A_nz":>5}')
for seed in SEEDS:
    p = Path(f'reports/phase34_{RUN_TAG}_seed{seed}/phase34_results.json')
    if not p.exists():
        print(f'{seed:>4} MISSING')
        continue
    d = json.loads(p.read_text())
    A = B = C = None
    for cond in ('baseline_static', 'phase3_reencode', 'phase3_phase4'):
        f = d['results'][cond][-1]
        cands = f.get('candidates_total', 0)
        cons_ = f.get('consolidations', 0)
        deaths = f.get('deaths_total', 0)
        dd = f.get('death_diag', {}).get('2', {})
        A_max = dd.get('inhibition_max', 0.0)
        A_nz  = dd.get('inhibition_nonzero', 0)
        print(f'{seed:>4} {cond:>16} {f["top1"]:>8.4f} {f["topk"]:>8.4f} {f["cap_t_05"]:>8.4f} '
              f'{cands:>6} {cons_:>5} {deaths:>6} {A_max:>8.3f} {A_nz:>5}')
        if cond == 'baseline_static':  A = f
        elif cond == 'phase3_reencode': B = f
        elif cond == 'phase3_phase4':   C = f
    if A and B and C:
        rows.append({'seed': seed,
                     'dR10_CA': C['topk']-A['topk'],
                     'dR10_CB': C['topk']-B['topk'],
                     'dtop1_CA': C['top1']-A['top1'],
                     'dcap_CA': C['cap_t_05']-A['cap_t_05']})

print()
print(f'=== aggregate across {len(rows)} seeds ===')
tcrit = {9: 2.262, 8: 2.306, 7: 2.365, 6: 2.447, 5: 2.571, 4: 2.776}
df = max(1, len(rows)-1)
tc = tcrit.get(df, 2.262)
for k in ['dR10_CA','dR10_CB','dtop1_CA','dcap_CA']:
    vals = [r[k] for r in rows]
    m = st.mean(vals); sd = st.stdev(vals) if len(vals)>1 else 0.0
    se = sd / (len(vals)**0.5) if len(vals)>1 else 0.0
    lo, hi = m - tc*se, m + tc*se
    pos = sum(1 for v in vals if v > 0)
    print(f'  {k}: mean={m:+.4f}, 95%CI [{lo:+.4f}, {hi:+.4f}], pos/total={pos}/{len(rows)}, '
          f'per-seed={[f"{v:+.4f}" for v in vals]}')